# E-commerce Checkout A/B Test
## Statistical Inference

### Executive Summary

The simplified single-page checkout increased 24-hour purchase conversion from **27.47%** in control to **29.63%** in treatment. The estimated effect is **+2.16 percentage points** (**+7.85% relative lift**).

A pre-specified two-sided two-proportion z-test gives **z = 2.968** and **p = 0.0030**. The **95% confidence interval for the absolute lift is +0.73 to +3.58 percentage points**, so the data support a positive treatment effect at the 5% significance level.

The point estimate exceeds the pre-specified business threshold of +1.0 percentage point, but the confidence interval still includes effects below that threshold. The treatment is therefore a promising launch candidate, not an automatic launch decision: guardrails, revenue, and operational evidence must also be reviewed.

### Analysis Plan

| Component | Pre-specified definition |
|---|---|
| Primary metric | Checkout purchase conversion rate |
| Analysis unit | User |
| Population | Eligible users with a first valid checkout exposure and a complete 24-hour observation window |
| Control | Existing multi-step checkout flow |
| Treatment | Simplified single-page checkout flow |
| Binary outcome | `reached_purchase = 1` when a user purchases after a qualifying payment attempt and within 24 hours of first valid exposure |
| Estimand | Treatment conversion minus control conversion |
| Significance level | 0.05 |
| Test | Two-sided two-proportion z-test |

Let `p_t` and `p_c` be treatment and control purchase-conversion probabilities in the mature exposed-user population.

- **Null hypothesis:** `H0: p_t - p_c = 0`
- **Alternative hypothesis:** `H1: p_t - p_c != 0`

The experiment design specified a two-sided test because the new checkout could help or harm conversion. The test direction is not changed after observing a positive result.

In [1]:
from pathlib import Path
from math import sqrt
from statistics import NormalDist
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = None

for path in [START_DIR, *START_DIR.parents]:
    if (
        path.name == "week2_checkout_experiment"
        and (path / "data" / "raw").is_dir()
    ):
        PROJECT_ROOT = path
        break

    candidate = path / "week2_checkout_experiment"
    if (candidate / "data" / "raw").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the week2_checkout_experiment directory."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
FUNNEL_SQL_PATH = PROJECT_ROOT / "sql" / "02_funnel_analysis.sql"

print("Project root:", PROJECT_ROOT.name)
print("Reused SQL definition:", FUNNEL_SQL_PATH.relative_to(PROJECT_ROOT))

Project root: week2_checkout_experiment
Reused SQL definition: sql/02_funnel_analysis.sql


## 1. Rebuild the Canonical Analysis Population

The notebook loads the versioned raw CSV files into an in-memory SQLite database and executes the canonical funnel SQL unchanged. This guarantees that the statistical test uses the same eligibility, exposure, maturity, event-deduplication, and ordered-funnel rules as the descriptive analysis.

In [2]:
conn = sqlite3.connect(":memory:")

raw_table_audit = []
for table_name in ["users", "experiment_assignments", "events", "orders"]:
    table = pd.read_csv(RAW_DATA_DIR / f"{table_name}.csv")
    table.to_sql(table_name, conn, index=False, if_exists="replace")
    raw_table_audit.append({"table_name": table_name, "rows": len(table)})

conn.executescript(FUNNEL_SQL_PATH.read_text(encoding="utf-8"))

print(pd.DataFrame(raw_table_audit).to_string(index=False))
print("\nCanonical user-level funnel rebuilt successfully.")

            table_name  rows
                 users 20000
experiment_assignments 20060
                events 36813
                orders  5043

Canonical user-level funnel rebuilt successfully.


## 2. Validate the Statistical Analysis Grain

The z-test requires one independent binary outcome per analysis unit. The checks below confirm one row per mature exposed user, a single experiment group per row, and a valid binary purchase flag.

In [3]:
analysis_data = pd.read_sql_query(
    """
    SELECT
        user_id,
        experiment_group,
        reached_purchase,
        user_type,
        device_at_exposure,
        traffic_source_at_exposure
    FROM user_level_funnel
    ORDER BY user_id;
    """,
    conn,
)

expected_group_sizes = {"control": 7_754, "treatment": 7_713}
expected_purchases = {"control": 2_130, "treatment": 2_285}

assert len(analysis_data) == 15_467
assert analysis_data["user_id"].nunique() == len(analysis_data)
assert set(analysis_data["experiment_group"]) == {"control", "treatment"}
assert set(analysis_data["reached_purchase"]) <= {0, 1}
assert not analysis_data["reached_purchase"].isna().any()

actual_group_sizes = analysis_data.groupby("experiment_group").size().to_dict()
actual_purchases = (
    analysis_data.groupby("experiment_group")["reached_purchase"].sum().to_dict()
)
assert actual_group_sizes == expected_group_sizes
assert actual_purchases == expected_purchases

grain_audit = pd.DataFrame(
    {
        "check": [
            "analysis rows",
            "unique users",
            "duplicate users",
            "missing outcomes",
        ],
        "value": [
            len(analysis_data),
            analysis_data["user_id"].nunique(),
            analysis_data["user_id"].duplicated().sum(),
            analysis_data["reached_purchase"].isna().sum(),
        ],
    }
)
print(grain_audit.to_string(index=False))

           check  value
   analysis rows  15467
    unique users  15467
 duplicate users      0
missing outcomes      0


## 3. Primary Metric by Experiment Group

In [4]:
group_summary = (
    analysis_data.groupby("experiment_group", as_index=False)
    .agg(
        users=("user_id", "size"),
        purchases=("reached_purchase", "sum"),
    )
    .sort_values("experiment_group")
    .reset_index(drop=True)
)
group_summary["non_purchases"] = (
    group_summary["users"] - group_summary["purchases"]
)
group_summary["conversion_rate"] = (
    group_summary["purchases"] / group_summary["users"]
)

group_summary_display = group_summary.copy()
group_summary_display["conversion_rate"] = group_summary_display[
    "conversion_rate"
].map("{:.4%}".format)
print(group_summary_display.to_string(index=False))

experiment_group  users  purchases  non_purchases conversion_rate
         control   7754       2130           5624        27.4697%
       treatment   7713       2285           5428        29.6253%


## 4. Two-Proportion Z-Test

For the hypothesis test, the null assumes a common conversion probability, so the test standard error uses the pooled rate:

`SE_null = sqrt(p_pool * (1 - p_pool) * (1/n_t + 1/n_c))`

The z-statistic is:

`z = (p_t - p_c) / SE_null`

For the confidence interval around the observed difference, the standard error is unpooled:

`SE_CI = sqrt(p_t*(1-p_t)/n_t + p_c*(1-p_c)/n_c)`

Using the pooled SE for the test and the unpooled SE for the confidence interval makes the inferential target explicit.

In [5]:
by_group = group_summary.set_index("experiment_group")

n_control = int(by_group.loc["control", "users"])
x_control = int(by_group.loc["control", "purchases"])
n_treatment = int(by_group.loc["treatment", "users"])
x_treatment = int(by_group.loc["treatment", "purchases"])

p_control = x_control / n_control
p_treatment = x_treatment / n_treatment
absolute_lift = p_treatment - p_control
relative_lift = absolute_lift / p_control

alpha = 0.05
normal = NormalDist()
pooled_rate = (x_control + x_treatment) / (n_control + n_treatment)
pooled_se = sqrt(
    pooled_rate
    * (1 - pooled_rate)
    * (1 / n_control + 1 / n_treatment)
)
unpooled_se = sqrt(
    p_control * (1 - p_control) / n_control
    + p_treatment * (1 - p_treatment) / n_treatment
)

z_statistic = absolute_lift / pooled_se
p_value = 2 * normal.cdf(-abs(z_statistic))
z_critical = normal.inv_cdf(1 - alpha / 2)
ci_lower = absolute_lift - z_critical * unpooled_se
ci_upper = absolute_lift + z_critical * unpooled_se
reject_null = p_value < alpha

inference_summary = pd.DataFrame(
    {
        "metric": [
            "Control conversion",
            "Treatment conversion",
            "Absolute lift (treatment - control)",
            "Relative lift",
            "Pooled standard error under H0",
            "Unpooled standard error for CI",
            "Z-statistic",
            "Two-sided p-value",
            "95% CI for absolute lift",
            "Decision at alpha = 0.05",
        ],
        "result": [
            f"{p_control:.2%}",
            f"{p_treatment:.2%}",
            f"{absolute_lift * 100:+.2f} pp",
            f"{relative_lift:+.2%}",
            f"{pooled_se * 100:.3f} pp",
            f"{unpooled_se * 100:.3f} pp",
            f"{z_statistic:.3f}",
            f"{p_value:.4f}",
            f"[{ci_lower * 100:+.2f} pp, {ci_upper * 100:+.2f} pp]",
            "Reject H0" if reject_null else "Fail to reject H0",
        ],
    }
)
print(inference_summary.to_string(index=False))

                             metric               result
                 Control conversion               27.47%
               Treatment conversion               29.63%
Absolute lift (treatment - control)             +2.16 pp
                      Relative lift               +7.85%
     Pooled standard error under H0             0.726 pp
     Unpooled standard error for CI             0.726 pp
                        Z-statistic                2.968
                  Two-sided p-value               0.0030
           95% CI for absolute lift [+0.73 pp, +3.58 pp]
           Decision at alpha = 0.05            Reject H0


### Statistical Significance

Because `p = 0.0030 < 0.05`, the null hypothesis of equal purchase conversion is rejected. The 95% confidence interval is entirely above zero, providing evidence that the simplified checkout increased conversion in the analyzed mature exposed-user population.

This does **not** mean there is a 0.3% probability that the null hypothesis is true. The p-value is the probability, assuming no true treatment-control difference and the model assumptions hold, of observing a standardized difference at least as extreme as this one.

### Practical and Business Significance

The point estimate of **+2.16 percentage points** is above the pre-specified **+1.0 percentage-point** business threshold and corresponds to a **+7.85% relative lift**, which is commercially meaningful for a checkout flow. However, the confidence interval ranges from **+0.73 to +3.58 percentage points**. It establishes a positive effect, but it does not establish with 95% confidence that the true effect exceeds +1.0 percentage point.

## 5. Assumption and Design Diagnostics

A valid p-value depends on the experiment and analysis process, not only the formula. The following diagnostics evaluate sample allocation, observed covariate balance, independence, and the large-sample approximation.

In [6]:
# Sample ratio mismatch diagnostic for the planned 50/50 allocation.
total_users = n_control + n_treatment
expected_treatment_share = 0.50
expected_treatment_users = total_users * expected_treatment_share
srm_se = sqrt(
    total_users
    * expected_treatment_share
    * (1 - expected_treatment_share)
)
srm_z = (n_treatment - expected_treatment_users) / srm_se
srm_p_value = 2 * normal.cdf(-abs(srm_z))

allocation_diagnostic = pd.DataFrame(
    {
        "control_users": [n_control],
        "treatment_users": [n_treatment],
        "treatment_share": [f"{n_treatment / total_users:.2%}"],
        "srm_z": [round(srm_z, 3)],
        "srm_p_value": [round(srm_p_value, 4)],
    }
)
print("Sample ratio diagnostic")
print(allocation_diagnostic.to_string(index=False))

# Compact descriptive balance check for baseline and exposure attributes.
balance_rows = []
for column in ["user_type", "device_at_exposure", "traffic_source_at_exposure"]:
    proportions = pd.crosstab(
        analysis_data[column],
        analysis_data["experiment_group"],
        normalize="columns",
    )
    max_gap_pp = (proportions["treatment"] - proportions["control"]).abs().max() * 100
    balance_rows.append(
        {"attribute": column, "maximum_absolute_category_gap_pp": max_gap_pp}
    )

balance_diagnostics = pd.DataFrame(balance_rows)
balance_diagnostics["maximum_absolute_category_gap_pp"] = balance_diagnostics[
    "maximum_absolute_category_gap_pp"
].round(3)
print("\nDescriptive covariate balance")
print(balance_diagnostics.to_string(index=False))

Sample ratio diagnostic
 control_users  treatment_users treatment_share  srm_z  srm_p_value
          7754             7713          49.87%  -0.33       0.7416

Descriptive covariate balance
                 attribute  maximum_absolute_category_gap_pp
                 user_type                             0.862
        device_at_exposure                             0.410
traffic_source_at_exposure                             0.736


In [7]:
# Expected cell counts under H0 should be comfortably above 5.
sample_size_checks = group_summary[["experiment_group", "users"]].copy()
sample_size_checks["expected_purchases_under_h0"] = (
    sample_size_checks["users"] * pooled_rate
)
sample_size_checks["expected_non_purchases_under_h0"] = (
    sample_size_checks["users"] * (1 - pooled_rate)
)
sample_size_checks["minimum_expected_cell"] = sample_size_checks[
    ["expected_purchases_under_h0", "expected_non_purchases_under_h0"]
].min(axis=1)
sample_size_checks["normal_approximation_passes"] = (
    sample_size_checks["minimum_expected_cell"] >= 5
)

assert sample_size_checks["normal_approximation_passes"].all()

for column in [
    "expected_purchases_under_h0",
    "expected_non_purchases_under_h0",
    "minimum_expected_cell",
]:
    sample_size_checks[column] = sample_size_checks[column].round(1)

print(sample_size_checks.to_string(index=False))

experiment_group  users  expected_purchases_under_h0  expected_non_purchases_under_h0  minimum_expected_cell  normal_approximation_passes
         control   7754                       2213.4                           5540.6                 2213.4                         True
       treatment   7713                       2201.6                           5511.4                 2201.6                         True


### Diagnostic Interpretation

- **Randomization and allocation:** Assignment was generated at the user level. The mature sample is 49.87% treatment, and the sample-ratio diagnostic (`p = 0.7416`) shows no evidence of a departure from the planned 50/50 allocation. User type, device, and traffic-source category shares differ by less than one percentage point at most. These checks can reveal anomalies but cannot prove that randomization was implemented correctly.
- **Independence:** The analysis has exactly one row and one binary outcome per randomized user. It assumes users do not influence one another and that one user's treatment does not change another user's purchase behavior.
- **Sample size:** All expected success and failure counts are far above 5, so the normal approximation is appropriate.
- **Consistent measurement:** Both groups use identical eligibility rules, event deduplication, ordered purchase logic, and 24-hour maturity windows.
- **Analysis population:** The estimand is explicitly for eligible, mature checkout-exposed users, not all assigned users or all site visitors. This triggered analysis is appropriate only if assignment cannot affect whether exposure is logged; otherwise conditioning on post-assignment exposure could introduce selection bias.
- **Analysis discipline:** The primary metric, two-sided hypothesis, and decision threshold were defined before primary inference. In a live experiment, unplanned repeated peeking or stopping would require sequential-testing adjustments.

## 6. Sensitivity: Approximate Minimum Detectable Effect

As a compact design diagnostic, the existing sample size is translated into an approximate minimum detectable effect (MDE) for a two-sided 5% test with 80% power, using the control conversion rate as the planning baseline. This is not used to replace the confidence interval or to claim retrospective proof of power.

In [8]:
target_power = 0.80
z_power = normal.inv_cdf(target_power)
planning_se = sqrt(
    p_control
    * (1 - p_control)
    * (1 / n_control + 1 / n_treatment)
)
approximate_mde = (z_critical + z_power) * planning_se

mde_summary = pd.DataFrame(
    {
        "alpha_two_sided": [alpha],
        "target_power": [target_power],
        "baseline_conversion": [f"{p_control:.2%}"],
        "approximate_mde": [f"{approximate_mde * 100:.2f} pp"],
        "mde_relative_to_baseline": [f"{approximate_mde / p_control:.2%}"],
    }
)
print(mde_summary.to_string(index=False))

 alpha_two_sided  target_power baseline_conversion approximate_mde mde_relative_to_baseline
            0.05           0.8              27.47%         2.01 pp                    7.32%


With approximately 7,700 users per group, the design has an approximate 80%-power MDE of **2.01 percentage points** around the 27.47% control baseline. The observed +2.16-point lift is slightly larger than this design sensitivity. The experiment was not designed to estimate a +1.0-point effect with the same precision, which is consistent with the confidence interval extending below the business threshold.

## 7. Business Translation

In [9]:
reference_exposed_users = 100_000
incremental_purchases = absolute_lift * reference_exposed_users
incremental_ci_lower = ci_lower * reference_exposed_users
incremental_ci_upper = ci_upper * reference_exposed_users

print(f"Per {reference_exposed_users:,} comparable mature checkout-exposed users:")
print(f"Point estimate: {incremental_purchases:,.0f} incremental purchases")
print(
    "95% CI: "
    f"{incremental_ci_lower:,.0f} to {incremental_ci_upper:,.0f} "
    "incremental purchases"
)

Per 100,000 comparable mature checkout-exposed users:
Point estimate: 2,156 incremental purchases
95% CI: 732 to 3,579 incremental purchases


The estimate translates to about **2,156 additional purchases per 100,000 comparable mature checkout-exposed users**, with a 95% confidence interval of approximately **732 to 3,579**. This scales the conversion effect without inventing revenue assumptions. A financial recommendation would additionally require traffic forecasts, order value, margin, implementation cost, and guardrail results.

## Final Decision Readout

**Statistical conclusion:** Reject the null hypothesis at `alpha = 0.05`. The treatment produced a statistically significant positive difference in 24-hour purchase conversion among eligible mature checkout-exposed users.

**Business conclusion:** The estimated +2.16 percentage-point lift is practically meaningful and exceeds the pre-specified point-estimate threshold. However, the lower confidence bound is +0.73 percentage points, so the experiment does not establish with 95% confidence that the true lift exceeds +1.0 point.

**Recommendation:** Treat the simplified checkout as a strong launch candidate, conditional on neutral or favorable payment-failure, checkout-error, revenue-per-exposed-user, refund, and cancellation guardrails. If the +1.0-point threshold must be met with high confidence, continue the experiment or collect a larger sample.

### Analytical Summary

> The experiment randomized at the user level and analyzed one binary purchase outcome per eligible checkout-exposed user using the same 24-hour window for both groups. Control converted at 27.47% and treatment at 29.63%, an absolute lift of 2.16 percentage points and a relative lift of 7.85%. A pre-specified two-sided two-proportion z-test produced z = 2.968 and p = 0.003, with a 95% confidence interval of +0.73 to +3.58 percentage points. The result is statistically significant and the point estimate is commercially meaningful, but the launch decision should not rely on the p-value alone: the interval includes effects below the +1-point threshold, and guardrails and revenue impact still require review.

In [10]:
conn.close()
print("In-memory SQLite connection closed.")

In-memory SQLite connection closed.
